# Part 3

In [1]:
REPO_URL = "https://github.com/SushOS/VLM-Extraction-Reasoning.git"
REPO_DIR = "VLM-Extraction-Reasoning"

In [2]:
!git clone {REPO_URL}

fatal: destination path 'VLM-Extraction-Reasoning' already exists and is not an empty directory.


In [3]:
%cd {REPO_DIR}

/content/VLM-Extraction-Reasoning


In [4]:
!pip install -r requirements.txt

In [6]:
# Prepare the real CORD benchmark subset
# !python prepare_cord.py --split validation --limit 32 --output-dir data/cord

!python prepare_cord.py --split validation --limit 10 --output-dir data/cord

README.md: 100% 27.0/27.0 [00:00<00:00, 121kB/s]
dataset_infos.json: 1.05kB [00:00, 3.01MB/s]
data/train-00000-of-00004-b4aaeceff1d90e(…): 100% 490M/490M [00:06<00:00, 76.2MB/s]
data/train-00001-of-00004-7dbbe248962764(…): 100% 441M/441M [00:02<00:00, 183MB/s]
data/train-00002-of-00004-688fe1305a55e5(…): 100% 444M/444M [00:02<00:00, 199MB/s]
data/train-00003-of-00004-2d0cd200555ed7(…): 100% 456M/456M [00:03<00:00, 151MB/s]
data/validation-00000-of-00001-cc3c5779f(…): 100% 242M/242M [00:02<00:00, 109MB/s]
data/test-00000-of-00001-9c204eb3f4e1179(…): 100% 234M/234M [00:01<00:00, 129MB/s]
Generating train split: 100% 800/800 [00:16<00:00, 47.29 examples/s]
Generating validation split: 100% 100/100 [00:05<00:00, 17.82 examples/s]
Generating test split: 100% 100/100 [00:01<00:00, 80.21 examples/s]


In [7]:
# compare both VLMs on cord receipts
!python model_compare.py --input data/cord/images --ground-truth-dir data/cord/ground_truth --task-name cord_receipt

preprocessor_config.json: 100% 350/350 [00:00<00:00, 2.07MB/s]
chat_template.json: 1.05kB [00:00, 3.37MB/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
config.json: 1.37kB [00:00, 4.53MB/s]
tokenizer_config.json: 5.70kB [00:00, 21.6MB/s]
vocab.json: 2.78MB [00:00, 81.3MB/s]
merges.txt: 1.67MB [00:00, 119MB/s]
tokenizer.json: 7.03MB [00:00, 129MB/s]
model.safetensors.index.json: 65.4kB [00:00, 99.6MB/s]
Fetching 2 files: 100% 2/2 [02:17<00:00, 68.51s/it]
Download complete: 100% 7.51G/7.51G [02:17<00:00, 54.7MB/s]
Loading weights: 100% 824/824 [00:30<00:00, 27.39it/s, Materializing param=model.visual.patch_embed.proj.weight]
generation_config.json: 100% 216/216 [00:00<00:00, 1.41MB/s]
The following generation flags are no

In [8]:
# Generate sample pdfs
!python generate_sample_pdfs.py

In [9]:
# Run end-to-end extraction on the sample PDFs
!python run_pipeline.py --input results/pdf_samples --output-dir results/pdf_outputs --task-name generic_document

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 824/824 [00:29<00:00, 27.95it/s, Materializing param=model.visual.patch_embed.proj.weight]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
# signature detection
!python run_pipeline.py --input results/pdf_samples --output-dir results/pdf_signature_outputs --task-name signature_check --ground-truth-dir data/pdf_ground_truth

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 824/824 [00:29<00:00, 27.77it/s, Materializing param=model.visual.patch_embed.proj.weight]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [11]:
# Form field filled/empty detection
!python run_pipeline.py --input results/pdf_samples --output-dir results/pdf_form_outputs --task-name form_fields --ground-truth-dir data/pdf_ground_truth

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100% 824/824 [00:29<00:00, 27.80it/s, Materializing param=model.visual.patch_embed.proj.weight]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [12]:
import torch
import gc

gc.collect()                 # clear Python garbage
torch.cuda.empty_cache()     # free unused GPU memory
torch.cuda.ipc_collect()     # clean inter-process memory

In [13]:
!python finetune_lora.py \
  --model-id HuggingFaceTB/SmolVLM2-2.2B-Instruct \
  --train-split train \
  --train-sample-count 128 \
  --eval-split validation \
  --eval-sample-count 32 \
  --max-image-size 768 \
  --gradient-accumulation-steps 4 \
  --learning-rate 1e-4 \
  --num-train-epochs 2 \
  --output-dir results/lora_adapter

Loading weights: 100% 657/657 [00:36<00:00, 18.23it/s, Materializing param=model.vision_model.post_layernorm.weight]
trainable params: 4,638,720 || all params: 2,251,423,600 || trainable%: 0.2060
{'loss': '15.38', 'grad_norm': '2.386', 'learning_rate': '9.375e-05', 'epoch': '0.1562'}
{'loss': '14.52', 'grad_norm': '3.439', 'learning_rate': '8.594e-05', 'epoch': '0.3125'}
{'loss': '13.87', 'grad_norm': '4.102', 'learning_rate': '7.813e-05', 'epoch': '0.4688'}
{'loss': '12.52', 'grad_norm': '3.932', 'learning_rate': '7.031e-05', 'epoch': '0.625'}
{'loss': '12.09', 'grad_norm': '4.192', 'learning_rate': '6.25e-05', 'epoch': '0.7812'}
{'loss': '11.16', 'grad_norm': '4.536', 'learning_rate': '5.469e-05', 'epoch': '0.9375'}
 50% 32/64 [09:02<08:57, 16.81s/it]
  0% 0/32 [00:00<?, ?it/s]
  6% 2/32 [00:01<00:21,  1.38it/s]
  9% 3/32 [00:02<00:29,  1.01s/it]
 12% 4/32 [00:04<00:32,  1.16s/it]
 16% 5/32 [00:05<00:33,  1.25s/it]
 19% 6/32 [00:07<00:33,  1.30s/it]
 22% 7/32 [00:08<00:33,  1.34s/it]

In [14]:
import torch
import gc

gc.collect()                 # clear Python garbage
torch.cuda.empty_cache()     # free unused GPU memory
torch.cuda.ipc_collect()     # clean inter-process memory

In [15]:
!python compare_finetuned.py \
  --input data/cord/images \
  --ground-truth-dir data/cord/ground_truth \
  --base-model smolvlm2_2b \
  --adapter-path results/lora_adapter \
  --task-name cord_receipt


Loading weights: 100% 657/657 [00:37<00:00, 17.58it/s, Materializing param=model.vision_model.post_layernorm.weight]
Loading weights: 100% 657/657 [00:37<00:00, 17.64it/s, Materializing param=model.vision_model.post_layernorm.weight]


In [16]:
!zip -r results.zip results/

  adding: results/ (stored 0%)
  adding: results/compare_smolvlm2_2b/ (stored 0%)
  adding: results/compare_smolvlm2_2b/cord_validation_0002_page_1.json (deflated 65%)
  adding: results/compare_smolvlm2_2b/cord_validation_0000_page_1.json (deflated 71%)
  adding: results/compare_smolvlm2_2b/cord_validation_0006_page_1.json (deflated 71%)
  adding: results/compare_smolvlm2_2b/cord_validation_0007_page_1.json (deflated 63%)
  adding: results/compare_smolvlm2_2b/cord_validation_0004_page_1.json (deflated 69%)
  adding: results/compare_smolvlm2_2b/cord_validation_0008_page_1.json (deflated 66%)
  adding: results/compare_smolvlm2_2b/cord_validation_0003_page_1.json (deflated 70%)
  adding: results/compare_smolvlm2_2b/cord_validation_0009_page_1.json (deflated 67%)
  adding: results/compare_smolvlm2_2b/cord_validation_0005_page_1.json (deflated 87%)
  adding: results/compare_smolvlm2_2b/cord_validation_0001_page_1.json (deflated 72%)
  adding: results/compare_processed_smolvlm2_2b_finetuned/

In [17]:
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
!zip -r data.zip data/

  adding: data/ (stored 0%)
  adding: data/cord/ (stored 0%)
  adding: data/cord/ground_truth/ (stored 0%)
  adding: data/cord/ground_truth/cord_validation_0002_page_1.json (deflated 55%)
  adding: data/cord/ground_truth/cord_validation_0000_page_1.json (deflated 64%)
  adding: data/cord/ground_truth/cord_validation_0006_page_1.json (deflated 64%)
  adding: data/cord/ground_truth/cord_validation_0007_page_1.json (deflated 53%)
  adding: data/cord/ground_truth/cord_validation_0004_page_1.json (deflated 56%)
  adding: data/cord/ground_truth/cord_validation_0008_page_1.json (deflated 58%)
  adding: data/cord/ground_truth/cord_validation_0003_page_1.json (deflated 55%)
  adding: data/cord/ground_truth/cord_validation_0009_page_1.json (deflated 56%)
  adding: data/cord/ground_truth/cord_validation_0005_page_1.json (deflated 53%)
  adding: data/cord/ground_truth/cord_validation_0001_page_1.json (deflated 56%)
  adding: data/cord/metadata.csv (deflated 87%)
  adding: data/cord/images/ (stored

In [19]:
from google.colab import files
files.download("data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>